# Мультимодальность - когда зрение, слух и язык объединяются

**Роль:** Преподаватель по ML
**Уровень:** средний (основы PyTorch + NLP)
**Время прохождения:** ~120-150 минут

---

### Чему вы научитесь

После прохождения этого ноутбука вы:
- поймёте, **что такое мультимодальность** и почему это следующий шаг ИИ;
- разберёте **CLIP** - контрастивное обучение изображений и текста;
- реализуете **простой CLIP-подобный** энкодер с нуля на PyTorch;
- научитесь **генерировать описания** изображений (image captioning);
- создадите **VQA-систему** - визуальный вопрос-ответ;
- построите **поисковик изображений** по текстовому запросу с FAISS;
- запустите **FastAPI сервер** мультимодального поиска;
- исследуете параметры через **интерактивные виджеты**;
- визуализируете **cross-attention** - как зрение встречается с языком;
- познакомитесь с **распознаванием речи** и мультимодальными моделями;
- сравните **CLIP, BLIP, LLaVA, GPT-4V**.

### Принцип этого блокнота

> Вы - автор, не пользователь. Каждая строка видна. Каждое действие можно сломать и починить. Никаких "чёрных ящиков".

---

## План урока

| # | Шаг | Что делаем |
|---|-----|------------|
| 1 | Подготовка окружения | Установка и импорт библиотек |
| 2 | Что такое мультимодальность? | Объединение зрения, текста, аудио |
| 3 | CLIP: контрастивное обучение | Как изображения и текст выстраиваются в общее пространство |
| 4 | Простой CLIP с нуля | Энкодер изображений + текста + contrastive loss |
| 5 | Image Captioning | Генерация текста по изображению |
| 6 | Visual Question Answering | Ответы на вопросы по картинке |
| 7 | Поиск изображений по тексту | FAISS + эмбеддинги |
| 8 | FastAPI сервер мультимодального поиска | /search, /caption, /similarity |
| 9 | Интерактивный мультимодальный эксплорер | Виджеты: температура, порог, top_k |
| 10 | Cross-attention: зрение встречает язык | Визуализация карт внимания |
| 11 | Аудио + текст: основы распознавания речи | Простой runnable пример |
| 12 | Ландшафт мультимодальных моделей | CLIP, BLIP, LLaVA, GPT-4V |
| 13 | Практические задания | 5 упражнений |

---

---
## Шаг 1. Подготовка окружения

| Библиотека | Зачем |
|-----------|-------|
| **torch** | Фреймворк глубокого обучения |
| **torchvision** | Предобученные модели для изображений |
| **Pillow** | Работа с изображениями |
| **matplotlib** | Визуализация данных |
| **ipywidgets** | Интерактивные виджеты |
| **transformers** | Предобученные мультимодальные модели |
| **faiss-cpu** | Быстрый поиск по векторам |
| **fastapi** + **uvicorn** | REST API сервер |
| **pyngrok** | Публичный доступ к серверу |

Установим всё необходимое:


In [ ]:
# Устанавливаем необходимые библиотеки
!pip install -q torch torchvision  # Фреймворк глубокого обучения + модели зрения
!pip install -q transformers  # Предобученные модели HuggingFace
!pip install -q faiss-cpu  # Быстрый поиск по векторам
!pip install -q Pillow  # Работа с изображениями
!pip install -q matplotlib  # Визуализация
!pip install -q ipywidgets  # Интерактивные виджеты
!pip install -q fastapi uvicorn  # REST API сервер
!pip install -q pyngrok  # Публичный туннель к серверу
!pip install -q sentence-transformers  # Эмбеддинги предложений
!pip install -q requests  # HTTP запросы

In [ ]:
# Импортируем базовые библиотеки
import torch  # Основной фреймворк
import torch.nn as nn  # Нейронные слои
import torch.nn.functional as F  # Функции активации и потерь
import numpy as np  # Математические операции
import matplotlib.pyplot as plt  # Графики и визуализация
from PIL import Image  # Работа с изображениями
import io  # Потоки байтов
import json  # Работа с JSON
import random  # Генерация случайных чисел
from typing import List, Dict, Optional  # Типизация

# Настройка matplotlib для отображения в ноутбуке
%matplotlib inline

# Устанавливаем种子 для воспроизводимости
torch.manual_seed(42)  # Фиксируем случайность PyTorch
np.random.seed(42)  # Фиксируем случайность NumPy
random.seed(42)  # Фиксируем случайность Python

# Проверяем доступность GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # Выбираем устройство
print(f"Устройство: {device}")  # Выводим выбранное устройство
print(f"PyTorch версия: {torch.__version__}")  # Выводим версию PyTorch

---
## Шаг 2. Что такое мультимодальность?

**Модальность** - это тип данных, который воспринимает система:
- **Текст** - слова, предложения, документы
- **Изображения** - пиксели, визуальные паттерны
- **Аудио** - звуковые волны, речь, музыка
- **Видео** - последовательность кадров + аудио

**Мультимодальность** - это способность модели одновременно обрабатывать
несколько типов данных и находить связи между ними.

### Почему это важно?

1. **Люди мультимодальны** - мы видим, слышим, читаем одновременно
2. **Больше информации** - текст + картинка > только текст
3. **Новые задачи** - поиск по фото, описание изображений, VQA
4. **Лучшее понимание** - контекст из разных источников дополняет друг друга

### Ключевая идея: общее пространство эмбеддингов

Вместо того чтобы работать с каждой модальностью отдельно,
мы отображаем их в **одно и то же векторное пространство**.
Тогда похожие по смыслу текст и изображение окажутся рядом.


In [ ]:
# Визуализация: мультимодальное пространство эмбеддингов
fig, axes = plt.subplots(1, 2, figsize=(14, 6))  # Создаём 2 графика

# --- Левый график: раздельные пространства ---
ax1 = axes[0]  # Первый подграфик
ax1.set_title("Раздельные пространства\n(без мультимодальности)", fontsize=13, fontweight="bold")  # Заголовок

# Изображения в своём пространстве (синие точки)
img_x = [1, 2, 3, 4]  # Координаты X для изображений
img_y = [3, 1, 4, 2]  # Координаты Y для изображений
img_labels = ["кошка", "собака", "машина", "дерево"]  # Подписи
ax1.scatter(img_x, img_y, c="royalblue", s=150, zorder=5, label="Изображения")  # Синие точки

# Тексты в другом пространстве (красные точки)
txt_x = [6, 7, 8, 9]  # Координаты X для текстов
txt_y = [2, 4, 1, 3]  # Координаты Y для текстов
txt_labels = ["кот", "пёс", "авто", "дуб"]  # Подписи
ax1.scatter(txt_x, txt_y, c="crimson", s=150, zorder=5, label="Тексты")  # Красные точки

# Подписываем точки
for i, label in enumerate(img_labels):  # Для каждого изображения
    ax1.annotate(label, (img_x[i], img_y[i]), fontsize=9, ha="center", va="bottom")  # Подпись
for i, label in enumerate(txt_labels):  # Для каждого текста
    ax1.annotate(label, (txt_x[i], txt_y[i]), fontsize=9, ha="center", va="bottom")  # Подпись

# Разделяющая линия
ax1.axvline(x=5.5, color="gray", linestyle="--", alpha=0.5)  # Вертикальная линия-разделитель
ax1.text(3, 5, "Пространство\nизображений", ha="center", fontsize=10, color="royalblue")  # Метка слева
ax1.text(7.5, 5, "Пространство\nтекстов", ha="center", fontsize=10, color="crimson")  # Метка справа
ax1.legend(fontsize=10)  # Легенда
ax1.set_xlim(0, 10)  # Пределы по X
ax1.set_ylim(0, 5.5)  # Пределы по Y

# --- Правый график: общее пространство ---
ax2 = axes[1]  # Второй подграфик
ax2.set_title("Общее пространство\n(мультимодальность!)", fontsize=13, fontweight="bold")  # Заголовок

# В общем пространстве похожие объекты рядом
shared_img_x = [1.5, 3.5, 6, 8]  # Координаты X для изображений
shared_img_y = [2, 3, 1, 4]  # Координаты Y для изображений
shared_txt_x = [1, 3, 6.5, 7.5]  # Координаты X для текстов
shared_txt_y = [1.5, 2.5, 1.5, 3.5]  # Координаты Y для текстов

ax2.scatter(shared_img_x, shared_img_y, c="royalblue", s=150, zorder=5, label="Изображения")  # Синие
ax2.scatter(shared_txt_x, shared_txt_y, c="crimson", s=150, zorder=5, label="Тексты")  # Красные

# Соединяем похожие пары пунктирными линиями
pairs = [(0, 0), (1, 1), (2, 2), (3, 3)]  # Индексы пар
for i, j in pairs:  # Для каждой пары
    ax2.plot([shared_img_x[i], shared_txt_x[j]], [shared_img_y[i], shared_txt_y[j]],
             "g--", alpha=0.5, linewidth=2)  # Зелёный пунктир

# Подписываем точки
for i, label in enumerate(img_labels):  # Для изображений
    ax2.annotate(label, (shared_img_x[i], shared_img_y[i]), fontsize=9, ha="center", va="bottom")  # Подпись
for i, label in enumerate(txt_labels):  # Для текстов
    ax2.annotate(label, (shared_txt_x[i], shared_txt_y[i]), fontsize=9, ha="center", va="bottom")  # Подпись

ax2.text(5, 5, "Похожие объекты\nрядом!", ha="center", fontsize=11,
         color="green", fontweight="bold")  # Общий комментарий
ax2.legend(fontsize=10)  # Легенда
ax2.set_xlim(0, 10)  # Пределы по X
ax2.set_ylim(0, 5.5)  # Пределы по Y

plt.tight_layout()  # Автоматическая подгонка
plt.show()  # Показываем график

---
## Шаг 3. CLIP - контрастивное обучение изображений и текста

**CLIP** (Contrastive Language-Image Pre-training) - модель от OpenAI (2021),
которая научилась связывать изображения и текст через **контрастивное обучение**.

### Как работает CLIP

1. **Image Encoder** (ViT или ResNet) превращает изображение в вектор
2. **Text Encoder** (Transformer) превращает текст в вектор
3. **Contrastive Loss** сближает правильные пары и раздвигает неправильные

### Контрастивный loss - ключевая идея

У нас есть N пар (изображение, текст). Для каждого изображения:
- Правильный текст - **позитивная пара** (должны быть ближе)
- Все остальные тексты - **негативные пары** (должны быть дальше)

Loss заставляет косинусное сходство правильных пар быть больше,
чем сходство неправильных пар на величину margin.


In [ ]:
# Визуализация: архитектура CLIP
fig, ax = plt.subplots(figsize=(14, 8))  # Создаём большой график
ax.set_xlim(0, 14)  # Пределы по X
ax.set_ylim(0, 10)  # Пределы по Y
ax.axis("off")  # Убираем оси
ax.set_title("Архитектура CLIP", fontsize=16, fontweight="bold", pad=20)  # Заголовок

# --- Левая часть: Image Encoder ---
# Изображение (вход)
img_box = plt.Rectangle((0.5, 7), 3, 2, fill=True, facecolor="lightyellow",
                          edgecolor="black", linewidth=2)  # Прямоугольник изображения
ax.add_patch(img_box)  # Добавляем на график
ax.text(2, 8, "Изображение\n(224x224x3)", ha="center", va="center", fontsize=10)  # Текст

# Стрелка вниз
ax.annotate("", xy=(2, 6.2), xytext=(2, 7), arrowprops=dict(arrowstyle="->", lw=2))  # Стрелка

# Image Encoder
enc_box1 = plt.Rectangle((0.5, 4), 3, 2, fill=True, facecolor="lightblue",
                           edgecolor="black", linewidth=2)  # Прямоугольник энкодера
ax.add_patch(enc_box1)  # Добавляем
ax.text(2, 5, "Image Encoder\n(ViT / ResNet)", ha="center", va="center", fontsize=10)  # Текст

# Стрелка вниз
ax.annotate("", xy=(2, 3.2), xytext=(2, 4), arrowprops=dict(arrowstyle="->", lw=2))  # Стрелка

# Image embedding
emb_box1 = plt.Rectangle((0.5, 1.5), 3, 1.5, fill=True, facecolor="lightskyblue",
                           edgecolor="black", linewidth=2)  # Прямоугольник эмбеддинга
ax.add_patch(emb_box1)  # Добавляем
ax.text(2, 2.25, "Image Embedding\n[d_1, d_2, ..., d_n]", ha="center", va="center", fontsize=9)  # Текст

# --- Правая часть: Text Encoder ---
# Текст (вход)
txt_box = plt.Rectangle((10.5, 7), 3, 2, fill=True, facecolor="lightyellow",
                          edgecolor="black", linewidth=2)  # Прямоугольник текста
ax.add_patch(txt_box)  # Добавляем
ax.text(12, 8, "Текст\n("кот на диване")", ha="center", va="center", fontsize=10)  # Текст

# Стрелка вниз
ax.annotate("", xy=(12, 6.2), xytext=(12, 7), arrowprops=dict(arrowstyle="->", lw=2))  # Стрелка

# Text Encoder
enc_box2 = plt.Rectangle((10.5, 4), 3, 2, fill=True, facecolor="lightcoral",
                           edgecolor="black", linewidth=2)  # Прямоугольник энкодера
ax.add_patch(enc_box2)  # Добавляем
ax.text(12, 5, "Text Encoder\n(Transformer)", ha="center", va="center", fontsize=10)  # Текст

# Стрелка вниз
ax.annotate("", xy=(12, 3.2), xytext=(12, 4), arrowprops=dict(arrowstyle="->", lw=2))  # Стрелка

# Text embedding
emb_box2 = plt.Rectangle((10.5, 1.5), 3, 1.5, fill=True, facecolor="salmon",
                           edgecolor="black", linewidth=2)  # Прямоугольник эмбеддинга
ax.add_patch(emb_box2)  # Добавляем
ax.text(12, 2.25, "Text Embedding\n[d_1, d_2, ..., d_n]", ha="center", va="center", fontsize=9)  # Текст

# --- Центр: Contrastive Loss ---
loss_box = plt.Rectangle((4.5, 3.5), 5, 2.5, fill=True, facecolor="lightgreen",
                           edgecolor="black", linewidth=2, linestyle="--")  # Прямоугольник loss
ax.add_patch(loss_box)  # Добавляем
ax.text(7, 5.3, "Contrastive Loss", ha="center", va="center", fontsize=12, fontweight="bold")  # Заголовок
ax.text(7, 4.5, "cos(img_i, txt_i) -> max", ha="center", va="center", fontsize=10, color="green")  # Позитив
ax.text(7, 3.9, "cos(img_i, txt_j) -> min", ha="center", va="center", fontsize=10, color="red")  # Негатив

# Стрелки от эмбеддингов к loss
ax.annotate("", xy=(4.5, 4.75), xytext=(3.5, 2.25), arrowprops=dict(arrowstyle="->", lw=2, color="blue"))  # От img
ax.annotate("", xy=(9.5, 4.75), xytext=(10.5, 2.25), arrowprops=dict(arrowstyle="->", lw=2, color="red"))  # От txt

# Подписи внизу
ax.text(7, 0.5, "Цель: одинаковые по смыслу изображение и текст -> похожие векторы",
        ha="center", fontsize=11, style="italic", color="darkgreen")  # Итоговый вывод

plt.tight_layout()  # Подгонка
plt.show()  # Показываем

---
## Шаг 4. Простой CLIP-подобный модель с нуля

Мы реализуем упрощённую версию CLIP на PyTorch:
- **SimpleImageEncoder** -小型 CNN для изображений
- **SimpleTextEncoder** - embedding + LSTM для текста
- **ContrastiveLoss** - InfoNCE loss
- Полный цикл обучения на синтетических данных

Каждый компонент написан вручную - никаких чёрных ящиков!


In [ ]:
# Простой энкодер изображений на CNN
class SimpleImageEncoder(nn.Module):
    # Энкодер превращает изображение 3x64x64 в вектор размером embed_dim
    def __init__(self, embed_dim: int = 64):
        super().__init__()  # Инициализация родительского класса
        # Сверточные слои: уменьшаем пространственное разрешение, увеличиваем каналы
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, stride=2, padding=1)  # 3->16 каналов, 64->32
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, stride=2, padding=1)  # 16->32 каналов, 32->16
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, stride=2, padding=1)  # 32->64 канала, 16->8
        self.pool = nn.AdaptiveAvgPool2d((1, 1))  # Глобальный пулинг до 1x1
        self.fc = nn.Linear(64, embed_dim)  # Линейный слой в embed_dim
        self.relu = nn.ReLU()  # Активация

    def forward(self, x):
        # x: [batch, 3, 64, 64]
        x = self.relu(self.conv1(x))  # Свертка 1 + активация
        x = self.relu(self.conv2(x))  # Свертка 2 + активация
        x = self.relu(self.conv3(x))  # Свертка 3 + активация
        x = self.pool(x)  # Глобальный пулинг: [batch, 64, 1, 1]
        x = x.view(x.size(0), -1)  # Вытягиваем: [batch, 64]
        x = self.fc(x)  # Проекция: [batch, embed_dim]
        return F.normalize(x, dim=-1)  # L2-нормализация: векторы на единичной сфере

# Создаём тестовое изображение и проверяем размерность
test_img = torch.randn(2, 3, 64, 64)  # Батч из 2 случайных изображений
img_encoder = SimpleImageEncoder(embed_dim=64)  # Создаём энкодер
test_output = img_encoder(test_img)  # Пропускаем через энкодер
print(f"Вход: {test_img.shape}")  # Размер входа
print(f"Выход: {test_output.shape}")  # Размер выхода
print(f"Норма вектора: {test_output[0].norm().item():.4f}")  # Должна быть ~1.0

In [ ]:
# Простой текстовый энкодер на LSTM
class SimpleTextEncoder(nn.Module):
    # Энкодер превращает последовательность токенов в вектор размером embed_dim
    def __init__(self, vocab_size: int = 1000, embed_dim: int = 64, hidden_dim: int = 128):
        super().__init__()  # Инициализация родительского класса
        self.embedding = nn.Embedding(vocab_size, hidden_dim)  # Слой эмбеддингов слов
        self.lstm = nn.LSTM(hidden_dim, hidden_dim, batch_first=True)  # LSTM слой
        self.fc = nn.Linear(hidden_dim, embed_dim)  # Проекция в общее пространство

    def forward(self, x):
        # x: [batch, seq_len] - последовательность индексов токенов
        emb = self.embedding(x)  # Эмбеддинги: [batch, seq_len, hidden_dim]
        output, (hidden, _) = self.lstm(emb)  # LSTM: hidden - [1, batch, hidden_dim]
        hidden = hidden.squeeze(0)  # Убираем размерность слоя: [batch, hidden_dim]
        out = self.fc(hidden)  # Проекция: [batch, embed_dim]
        return F.normalize(out, dim=-1)  # L2-нормализация

# Тестируем текстовый энкодер
test_text = torch.randint(0, 1000, (2, 10))  # Батч из 2 последовательностей по 10 токенов
txt_encoder = SimpleTextEncoder(vocab_size=1000, embed_dim=64)  # Создаём энкодер
test_txt_output = txt_encoder(test_text)  # Пропускаем через энкодер
print(f"Вход: {test_text.shape}")  # Размер входа
print(f"Выход: {test_txt_output.shape}")  # Размер выхода
print(f"Норма вектора: {test_txt_output[0].norm().item():.4f}")  # Должна быть ~1.0

In [ ]:
# Контрастивный loss (InfoNCE) - сердце CLIP
def contrastive_loss(image_features, text_features, temperature=0.07):
    # image_features: [batch, embed_dim] - нормализованные векторы изображений
    # text_features: [batch, embed_dim] - нормализованные векторы текстов
    # temperature: параметр шкалирования (чем меньше, тем острее распределение)

    batch_size = image_features.shape[0]  # Размер батча

    # Вычисляем матрицу косинусных сходств: [batch, batch]
    # (i, j) элемент = сходство между i-м изображением и j-м текстом
    logits = (image_features @ text_features.T) / temperature  # Матрица сходств

    # Правильные пары - на диагонали (i-е изображение с i-м текстом)
    labels = torch.arange(batch_size, device=image_features.device)  # [0, 1, 2, ...]

    # Loss для изображений: для каждого img найти правильный текст
    loss_img = F.cross_entropy(logits, labels)  # Cross-entropy по строкам

    # Loss для текстов: для каждого txt найти правильное изображение
    loss_txt = F.cross_entropy(logits.T, labels)  # Cross-entropy по столбцам

    # Средний loss (симметричный)
    total_loss = (loss_img + loss_txt) / 2  # Усредняем оба направления

    return total_loss  # Возвращаем общий loss

# Тестируем loss на случайных данных
dummy_img = F.normalize(torch.randn(4, 64), dim=-1)  # 4 случайных вектора изображений
dummy_txt = F.normalize(torch.randn(4, 64), dim=-1)  # 4 случайных вектора текстов
loss = contrastive_loss(dummy_img, dummy_txt)  # Вычисляем loss
print(f"Loss на случайных данных: {loss.item():.4f}")  # Должно быть ~ln(4)=1.386

In [ ]:
# Полная модель SimpleCLIP - объединяем оба энкодера
class SimpleCLIP(nn.Module):
    def __init__(self, embed_dim=64, vocab_size=1000):
        super().__init__()  # Инициализация
        self.image_encoder = SimpleImageEncoder(embed_dim)  # Энкодер изображений
        self.text_encoder = SimpleTextEncoder(vocab_size, embed_dim)  # Энкодер текстов
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / 0.07))  # Логарифм температуры

    def forward(self, images, text_tokens):
        # images: [batch, 3, 64, 64]
        # text_tokens: [batch, seq_len]
        image_features = self.image_encoder(images)  # Векторы изображений
        text_features = self.text_encoder(text_tokens)  # Векторы текстов
        # Температура как обучаемый параметр (exp для положительности)
        temperature = self.logit_scale.exp().clamp(max=100)  # Ограничиваем сверху
        loss = contrastive_loss(image_features, text_features, temperature)  # Loss
        return loss, image_features, text_features  # Возвращаем loss и векторы

# Создаём модель
model = SimpleCLIP(embed_dim=64, vocab_size=1000).to(device)  # Инициализация модели
total_params = sum(p.numel() for p in model.parameters())  # Считаем параметры
print(f"Модель SimpleCLIP создана")  # Сообщение
print(f"Всего параметров: {total_params:,}")  # Количество параметров

In [ ]:
# Генерируем синтетические данные для обучения
def generate_synthetic_data(n_samples=200, img_size=64, seq_len=8, vocab_size=1000, n_classes=10):
    # Создаём данные, где каждому классу соответствует свой визуальный и текстовый паттерн
    images = []  # Список изображений
    texts = []  # Список текстов
    labels = []  # Список меток классов

    for i in range(n_samples):  # Для каждого примера
        cls = i % n_classes  # Класс (циклически)
        labels.append(cls)  # Сохраняем метку

        # Изображение: уникальный цвет и паттерн для каждого класса
        hue = cls / n_classes  # Оттенок
        img = torch.zeros(3, img_size, img_size)  # Пустое изображение
        # Канал R: зависит от класса
        img[0] = hue + torch.randn(img_size, img_size) * 0.1  # Красный канал + шум
        # Канал G: другой паттерн
        img[1] = (1 - hue) + torch.randn(img_size, img_size) * 0.1  # Зелёный канал + шум
        # Канал B: комбинация
        img[2] = torch.randn(img_size, img_size) * 0.1  # Синий канал (шум)
        # Добавляем уникальный паттерн: полоса на определённой позиции
        stripe_pos = int((cls / n_classes) * img_size)  # Позиция полосы
        img[:, stripe_pos:stripe_pos+4, :] += 1.0  # Горизонтальная полоса
        img = img.clamp(0, 1)  # Ограничиваем значения [0, 1]
        images.append(img)  # Добавляем в список

        # Текст: токены, начинающиеся с токена класса
        text_tokens = [cls * 100 + j for j in range(seq_len)]  # Токены привязаны к классу
        text_tokens = [min(t, vocab_size - 1) for t in text_tokens]  # Ограничиваем vocab
        texts.append(torch.tensor(text_tokens, dtype=torch.long))  # Добавляем как тензор

    # Стекаем в тензоры
    images_tensor = torch.stack(images).to(device)  # [n_samples, 3, 64, 64]
    texts_tensor = torch.stack(texts).to(device)  # [n_samples, seq_len]
    labels_tensor = torch.tensor(labels)  # [n_samples]

    return images_tensor, texts_tensor, labels_tensor  # Возвращаем данные

# Генерируем данные
images_data, texts_data, labels_data = generate_synthetic_data(n_samples=200)  # 200 примеров
print(f"Изображения: {images_data.shape}")  # Размерность изображений
print(f"Тексты: {texts_data.shape}")  # Размерность текстов
print(f"Метки: {labels_data.shape}")  # Размерность меток

In [ ]:
# Обучаем SimpleCLIP на синтетических данных
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)  # Оптимизатор Adam
n_epochs = 50  # Количество эпох
batch_size = 32  # Размер батча
losses_history = []  # История loss для графика

model.train()  # Режим обучения
for epoch in range(n_epochs):  # Для каждой эпохи
    # Перемешиваем данные
    perm = torch.randperm(images_data.shape[0])  # Случайная перестановка
    epoch_loss = 0.0  # Суммарный loss за эпоху
    n_batches = 0  # Счётчик батчей

    for i in range(0, len(perm), batch_size):  # По батчам
        idx = perm[i:i+batch_size]  # Индексы текущего батча
        batch_img = images_data[idx]  # Батч изображений
        batch_txt = texts_data[idx]  # Батч текстов

        loss, img_feat, txt_feat = model(batch_img, batch_txt)  # Прямой проход

        optimizer.zero_grad()  # Обнуляем градиенты
        loss.backward()  # Обратный проход
        optimizer.step()  # Обновление весов

        epoch_loss += loss.item()  # Накапливаем loss
        n_batches += 1  # Увеличиваем счётчик

    avg_loss = epoch_loss / n_batches  # Средний loss за эпоху
    losses_history.append(avg_loss)  # Сохраняем в историю

    if (epoch + 1) % 10 == 0:  # Каждые 10 эпох
        print(f"Эпоха {epoch+1}/{n_epochs}, Loss: {avg_loss:.4f}")  # Выводим прогресс

print("Обучение завершено!")  # Готово

In [ ]:
# Визуализация кривых обучения контрастивной модели
fig, ax = plt.subplots(figsize=(10, 5))  # Создаём график
ax.plot(range(1, n_epochs+1), losses_history, "b-", linewidth=2, label="Contrastive Loss")  # Кривая loss
ax.set_xlabel("Эпоха", fontsize=12)  # Подпись оси X
ax.set_ylabel("Loss (InfoNCE)", fontsize=12)  # Подпись оси Y
ax.set_title("Кривая обучения SimpleCLIP", fontsize=14, fontweight="bold")  # Заголовок
ax.legend(fontsize=11)  # Легенда
ax.grid(True, alpha=0.3)  # Сетка
plt.tight_layout()  # Подгонка
plt.show()  # Показываем

In [ ]:
# Визуализируем матрицу сходства после обучения
model.eval()  # Режим оценки
with torch.no_grad():  # Без градиентов
    # Берём 8 примеров из каждого класса
    vis_indices = []  # Индексы для визуализации
    for cls in range(8):  # 8 классов
        cls_idx = (labels_data == cls).nonzero(as_tuple=True)[0][:1]  # Первый пример класса
        if len(cls_idx) > 0:  # Если есть примеры
            vis_indices.append(cls_idx.item())  # Добавляем индекс

    vis_indices = vis_indices[:8]  # Берём до 8 примеров
    vis_img = images_data[vis_indices]  # Изображения для визуализации
    vis_txt = texts_data[vis_indices]  # Тексты для визуализации

    # Получаем эмбеддинги
    img_feat, txt_feat = model.image_encoder(vis_img), model.text_encoder(vis_txt)  # Векторы

    # Матрица сходства
    sim_matrix = (img_feat @ txt_feat.T).cpu().numpy()  # Косинусные сходства

# Строим тепловую карту
fig, ax = plt.subplots(figsize=(8, 7))  # Создаём график
im = ax.imshow(sim_matrix, cmap="RdYlGn", vmin=-1, vmax=1)  # Тепловая карта
class_labels = [f"Кл.{i}" for i in range(len(vis_indices))]  # Подписи
ax.set_xticks(range(len(vis_indices)))  # Деления X
ax.set_yticks(range(len(vis_indices)))  # Деления Y
ax.set_xticklabels(class_labels, fontsize=10)  # Подписи X
ax.set_yticklabels(class_labels, fontsize=10)  # Подписи Y
ax.set_xlabel("Текстовые эмбеддинги", fontsize=12)  # Подпись оси X
ax.set_ylabel("Визуальные эмбеддинги", fontsize=12)  # Подпись оси Y
ax.set_title("Матрица сходства: изображения vs тексты", fontsize=13, fontweight="bold")  # Заголовок

# Числа в ячейках
for i in range(len(vis_indices)):  # По строкам
    for j in range(len(vis_indices)):  # По столбцам
        color = "white" if abs(sim_matrix[i, j]) > 0.5 else "black"  # Цвет текста
        ax.text(j, i, f"{sim_matrix[i, j]:.2f}", ha="center", va="center",
                fontsize=9, color=color)  # Число в ячейке

plt.colorbar(im, ax=ax, label="Косинусное сходство")  # Цветовая шкала
plt.tight_layout()  # Подгонка
plt.show()  # Показываем

---
## Шаг 5. Image Captioning - генерация текста по изображению

**Image Captioning** - задача генерации текстового описания изображения.

Мы используем предобученную модель BLIP (Bootstrapping Language-Image Pre-training)
от Salesforce, которая умеет как captioning, так и VQA.


In [ ]:
# Загружаем предобученную модель BLIP для captioning
from transformers import BlipProcessor, BlipForConditionalGeneration  # Импортируем BLIP

# Загружаем процессор и модель
blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")  # Процессор
blip_model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")  # Модель
blip_model = blip_model.to(device)  # Переносим на GPU/CPU
blip_model.eval()  # Режим оценки
print("BLIP модель загружена!")  # Готово

In [ ]:
# Создаём тестовые изображения с помощью matplotlib
def create_test_image(scene_type="cat"):
    # Создаём простое изображение заданного типа
    fig, ax = plt.subplots(figsize=(3, 3))  # Маленький график
    ax.set_xlim(0, 10)  # Пределы X
    ax.set_ylim(0, 10)  # Пределы Y

    if scene_type == "cat":  # Сцена с котом
        # Тело кота
        cat_body = plt.Circle((5, 4), 2, color="orange")  # Оранжевый круг
        ax.add_patch(cat_body)  # Добавляем
        # Голова
        cat_head = plt.Circle((5, 7), 1.2, color="orange")  # Голова
        ax.add_patch(cat_head)  # Добавляем
        # Уши
        ax.plot([4, 4.3, 4.6], [7.8, 8.8, 7.8], "k-", linewidth=2)  # Левое ухо
        ax.plot([5.4, 5.7, 6], [7.8, 8.8, 7.8], "k-", linewidth=2)  # Правое ухо
        # Глаза
        ax.plot(4.5, 7.2, "ko", markersize=5)  # Левый глаз
        ax.plot(5.5, 7.2, "ko", markersize=5)  # Правый глаз
        # Диван
        sofa = plt.Rectangle((1, 1), 8, 2, color="brown", alpha=0.7)  # Диван
        ax.add_patch(sofa)  # Добавляем

    elif scene_type == "sunset":  # Закат
        # Небо
        ax.set_facecolor("orange")  # Оранжевое небо
        # Солнце
        sun = plt.Circle((5, 7), 1.5, color="yellow")  # Солнце
        ax.add_patch(sun)  # Добавляем
        # Горы
        mountain_x = [0, 3, 5, 7, 10]  # X-координаты гор
        mountain_y = [4, 6, 4.5, 5.5, 4]  # Y-координаты гор
        ax.fill(mountain_x, mountain_y, color="darkgreen")  # Заливка гор
        # Вода
        water = plt.Rectangle((0, 0), 10, 4, color="blue", alpha=0.5)  # Вода
        ax.add_patch(water)  # Добавляем

    elif scene_type == "city":  # Город
        ax.set_facecolor("lightblue")  # Голубое небо
        # Здания
        building1 = plt.Rectangle((1, 1), 2, 6, color="gray")  # Здание 1
        building2 = plt.Rectangle((4, 2), 2, 5, color="darkgray")  # Здание 2
        building3 = plt.Rectangle((7, 1), 2, 7, color="silver")  # Здание 3
        for b in [building1, building2, building3]:  # Для каждого здания
            ax.add_patch(b)  # Добавляем
        # Окна
        for x in [1.5, 2, 2.5]:  # Окна здания 1
            for y in [2, 3, 4, 5]:  # По этажам
                ax.add_patch(plt.Rectangle((x-0.15, y-0.15), 0.3, 0.3, color="yellow"))  # Окно

    ax.axis("off")  # Убираем оси
    plt.tight_layout()  # Подгонка

    # Сохраняем в буфер
    buf = io.BytesIO()  # Буфер в памяти
    fig.savefig(buf, format="png", bbox_inches="tight", pad_inches=0)  # Сохраняем
    plt.close(fig)  # Закрываем фигуру
    buf.seek(0)  # Перемещаем в начало
    img = Image.open(buf).convert("RGB")  # Открываем как PIL Image
    return img  # Возвращаем изображение

# Создаём и показываем тестовые изображения
scenes = ["cat", "sunset", "city"]  # Типы сцен
fig, axes = plt.subplots(1, 3, figsize=(12, 4))  # 3 графика
for i, scene in enumerate(scenes):  # Для каждой сцены
    img = create_test_image(scene)  # Создаём изображение
    axes[i].imshow(img)  # Показываем
    axes[i].set_title(scene, fontsize=12)  # Заголовок
    axes[i].axis("off")  # Убираем оси
plt.suptitle("Тестовые изображения", fontsize=14, fontweight="bold")  # Общий заголовок
plt.tight_layout()  # Подгонка
plt.show()  # Показываем

In [ ]:
# Генерируем описания (captions) для тестовых изображений
scenes = ["cat", "sunset", "city"]  # Типы сцен
scene_names = {"cat": "Кот на диване", "sunset": "Закат над горами", "city": "Городской пейзаж"}  # Русские названия

for scene in scenes:  # Для каждой сцены
    img = create_test_image(scene)  # Создаём изображение
    # Подготавливаем вход для модели
    inputs = blip_processor(img, return_tensors="pt").to(device)  # Токенизация
    # Генерируем описание
    with torch.no_grad():  # Без градиентов
        output = blip_model.generate(**inputs, max_new_tokens=50)  # Генерация
    # Декодируем результат
    caption = blip_processor.decode(output[0], skip_special_tokens=True)  # Текст описания
    print(f"Сцена '{scene_names[scene]}': {caption}")  # Выводим результат

---
## Шаг 6. Visual Question Answering (VQA)

**VQA** - задача ответа на вопрос по изображению.
Модель получает изображение и текстовый вопрос, и должна дать ответ.

Мы используем BLIP-VQA - ту же архитектуру, но настроенную для ответов на вопросы.


In [ ]:
# Загружаем модель BLIP для VQA
from transformers import BlipForQuestionAnswering  # Импортируем модель VQA

vqa_processor = BlipProcessor.from_pretrained("Salesforce/blip-vqa-base")  # Процессор VQA
vqa_model = BlipForQuestionAnswering.from_pretrained("Salesforce/blip-vqa-base")  # Модель VQA
vqa_model = vqa_model.to(device)  # На GPU/CPU
vqa_model.eval()  # Режим оценки
print("BLIP-VQA модель загружена!")  # Готово

In [ ]:
# Задаём вопросы к нашим тестовым изображениям
# Словарь: сцена -> список вопросов
questions = {
    "cat": ["what is the animal?", "what color is the animal?", "where is the animal sitting?"],
    "sunset": ["what time of day is it?", "what is in the sky?", "what color is the water?"],
    "city": ["how many buildings are there?", "what is the weather?", "what color are the windows?"]
}

for scene in scenes:  # Для каждой сцены
    img = create_test_image(scene)  # Создаём изображение
    print(f"\n=== Сцена: {scene_names[scene]} ===")  # Заголовок
    for question in questions[scene]:  # Для каждого вопроса
        # Подготавливаем вход: изображение + вопрос
        inputs = vqa_processor(img, question, return_tensors="pt").to(device)  # Токенизация
        # Генерируем ответ
        with torch.no_grad():  # Без градиентов
            output = vqa_model.generate(**inputs)  # Генерация ответа
        # Декодируем
        answer = vqa_processor.decode(output[0], skip_special_tokens=True)  # Текст ответа
        print(f"  Q: {question}")  # Вопрос
        print(f"  A: {answer}")  # Ответ

---
## Шаг 7. Поиск изображений по текстовому запросу

Теперь создадим поисковую систему: пользователь вводит текст,
а система находит наиболее релевантные изображения.

Мы используем **sentence-transformers** для эмбеддингов и **FAISS** для быстрого поиска.


In [ ]:
# Создаём мини-датасет изображений и описаний
from sentence_transformers import SentenceTransformer  # Импортируем модель эмбеддингов

# Создаём набор тестовых изображений с описаниями
dataset_items = []  # Список элементов датасета
descriptions = [  # Описания на английском (для модели)
    "a fluffy orange cat sitting on a brown sofa",
    "a beautiful sunset over mountains and lake",
    "tall gray buildings in a city skyline",
    "a red sports car parked on a street",
    "a green forest with tall pine trees",
    "a white dog running on the beach",
    "a plate of colorful sushi on a wooden table",
    "a yellow flower blooming in a garden",
    "a blue bicycle leaning against a wall",
    "a brown horse grazing in a meadow",
    "a purple butterfly on a pink flower",
    "a black cat sleeping on a windowsill",
    "a rainbow appearing after rain",
    "a small boat sailing on the ocean",
    "a cup of hot coffee with steam"
]

# Создаём изображения программно с помощью matplotlib
def create_scene_image(idx):
    # Создаём уникальное изображение по индексу
    fig, ax = plt.subplots(figsize=(2, 2))  # Маленький график
    np.random.seed(idx)  # Фиксируем случайность для воспроизводимости

    # Разные цветовые схемы для разных индексов
    colors = ["red", "blue", "green", "orange", "purple",
              "cyan", "magenta", "brown", "pink", "gold",
              "teal", "black", "violet", "navy", "coral"]  # 15 цветов
    bg_colors = ["lightyellow", "lightblue", "lightgreen", "wheat", "lavender",
                 "lightcyan", "mistyrose", "linen", "pink", "lemonchiffon",
                 "honeydew", "aliceblue", "thistle", "azure", "seashell"]  # 15 фонов

    ax.set_facecolor(bg_colors[idx % 15])  # Фон

    # Основная фигура
    shape = idx % 4  # Тип фигуры
    if shape == 0:  # Круг
        circle = plt.Circle((0.5, 0.5), 0.3, color=colors[idx % 15])  # Круг
        ax.add_patch(circle)  # Добавляем
    elif shape == 1:  # Квадрат
        rect = plt.Rectangle((0.2, 0.2), 0.6, 0.6, color=colors[idx % 15])  # Квадрат
        ax.add_patch(rect)  # Добавляем
    elif shape == 2:  # Треугольник
        triangle = plt.Polygon([[0.5, 0.8], [0.2, 0.2], [0.8, 0.2]],
                               color=colors[idx % 15])  # Треугольник
        ax.add_patch(triangle)  # Добавляем
    else:  # Звезда (два треугольника)
        tri1 = plt.Polygon([[0.5, 0.9], [0.2, 0.3], [0.8, 0.3]],
                           color=colors[idx % 15])  # Верхний треугольник
        tri2 = plt.Polygon([[0.5, 0.1], [0.2, 0.7], [0.8, 0.7]],
                           color=colors[idx % 15], alpha=0.7)  # Нижний треугольник
        ax.add_patch(tri1)  # Добавляем
        ax.add_patch(tri2)  # Добавляем

    ax.set_xlim(0, 1)  # Пределы X
    ax.set_ylim(0, 1)  # Пределы Y
    ax.axis("off")  # Убираем оси

    # Сохраняем в буфер
    buf = io.BytesIO()  # Буфер
    fig.savefig(buf, format="png", bbox_inches="tight", pad_inches=0, dpi=64)  # Сохраняем
    plt.close(fig)  # Закрываем
    buf.seek(0)  # В начало
    return Image.open(buf).convert("RGB")  # Как PIL Image

# Создаём датасет
images_dataset = []  # Список изображений
for i, desc in enumerate(descriptions):  # Для каждого описания
    img = create_scene_image(i)  # Создаём изображение
    images_dataset.append(img)  # Добавляем
    dataset_items.append({"id": i, "description": desc, "image": img})  # Сохраняем

print(f"Датасет создан: {len(dataset_items)} элементов")  # Сообщение

In [ ]:
# Строим индекс FAISS для поиска по текстовым описаниям
import faiss  # Импортируем FAISS

# Загружаем модель для текстовых эмбеддингов
st_model = SentenceTransformer("all-MiniLM-L6-v2")  # Лёгкая и быстрая модель
print("SentenceTransformer загружен!")  # Готово

# Вычисляем эмбеддинги для всех описаний
text_embeddings = st_model.encode(descriptions, convert_to_numpy=True)  # [15, 384]
print(f"Эмбеддинги: {text_embeddings.shape}")  # Размерность

# L2-нормализация для косинусного сходства
faiss.normalize_L2(text_embeddings)  # Нормализация на месте

# Создаём индекс FAISS (Inner Product = косинусное сходство после нормализации)
index = faiss.IndexFlatIP(text_embeddings.shape[1])  # Индекс для inner product
index.add(text_embeddings)  # Добавляем векторы
print(f"FAISS индекс создан: {index.ntotal} векторов")  # Готово

In [ ]:
# Функция поиска изображений по текстовому запросу
def search_images(query: str, top_k: int = 5):
    # query: текстовый запрос на английском
    # top_k: сколько результатов вернуть
    query_embedding = st_model.encode([query], convert_to_numpy=True)  # Эмбеддинг запроса
    faiss.normalize_L2(query_embedding)  # Нормализация
    scores, indices = index.search(query_embedding, top_k)  # Поиск в FAISS
    results = []  # Список результатов
    for score, idx in zip(scores[0], indices[0]):  # Для каждого результата
        results.append({
            "id": int(idx),  # Идентификатор
            "description": descriptions[idx],  # Описание
            "score": float(score)  # Оценка сходства
        })
    return results  # Возвращаем результаты

# Тестируем поиск
test_queries = ["a cute animal", "nature and landscape", "something to eat"]  # Тестовые запросы
for query in test_queries:  # Для каждого запроса
    print(f"\nЗапрос: '{query}'")  # Выводим запрос
    results = search_images(query, top_k=3)  # Ищем топ-3
    for r in results:  # Для каждого результата
        print(f"  [{r['score']:.3f}] {r['description']}")  # Выводим результат

In [ ]:
# Визуализация результатов поиска
def visualize_search(query: str, top_k: int = 5):
    # Показываем запрос и найденные изображения с оценками
    results = search_images(query, top_k)  # Получаем результаты
    n = len(results)  # Количество результатов
    fig, axes = plt.subplots(1, n, figsize=(3*n, 3))  # Создаём подграфики
    if n == 1:  # Если один результат
        axes = [axes]  # Оборачиваем в список
    for i, r in enumerate(results):  # Для каждого результата
        axes[i].imshow(images_dataset[r["id"]])  # Показываем изображение
        axes[i].set_title(f"Score: {r['score']:.3f}", fontsize=10)  # Оценка
        axes[i].axis("off")  # Убираем оси
    plt.suptitle(f"Запрос: '{query}'", fontsize=13, fontweight="bold")  # Заголовок
    plt.tight_layout()  # Подгонка
    plt.show()  # Показываем

# Визуализируем несколько запросов
visualize_search("a cute pet", top_k=4)  # Поиск милых питомцев
visualize_search("outdoor scenery", top_k=4)  # Поиск природы

---
## Шаг 8. FastAPI сервер мультимодального поиска

Теперь мы поднимем полноценный REST API сервер с тремя эндпоинтами:

| Эндпоинт | Метод | Описание |
|----------|-------|----------|
| `/search` | POST | Поиск изображений по текстовому запросу |
| `/caption` | POST | Генерация описания для загруженного изображения |
| `/similarity` | POST | Оценка сходства изображения и текста |

Сервер будет доступен через **ngrok** - публичный URL.


In [ ]:
# Сохраняем файл сервера
server_code = '''
import io  # Потоки байтов
import base64  # Кодирование Base64
import numpy as np  # Математика
from fastapi import FastAPI, HTTPException  # FastAPI
from fastapi.middleware.cors import CORSMiddleware  # CORS
from pydantic import BaseModel  # Модели данных
from PIL import Image  # Работа с изображениями
import torch  # PyTorch
import faiss  # Быстрый поиск
from sentence_transformers import SentenceTransformer  # Эмбеддинги
from transformers import BlipProcessor, BlipForConditionalGeneration  # BLIP

# Создаём приложение FastAPI
app = FastAPI(title="Multimodal Search API", version="1.0")  # Приложение

# Добавляем CORS для доступа из браузера
app.add_middleware(  # Middleware для CORS
    CORSMiddleware,
    allow_origins=["*"],  # Разрешаем все источники
    allow_methods=["*"],  # Разрешаем все методы
    allow_headers=["*"],  # Разрешаем все заголовки
)

# Загружаем модели при старте
st_model = SentenceTransformer("all-MiniLM-L6-v2")  # Модель эмбеддингов
blip_processor = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")  # BLIP процессор
blip_model = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base")  # BLIP модель
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")  # Устройство
blip_model = blip_model.to(device)  # Модель на устройство
blip_model.eval()  # Режим оценки

# Мини-датасет описаний
descriptions = [  # Описания изображений
    "a fluffy orange cat sitting on a brown sofa",
    "a beautiful sunset over mountains and lake",
    "tall gray buildings in a city skyline",
    "a red sports car parked on a street",
    "a green forest with tall pine trees",
    "a white dog running on the beach",
    "a plate of colorful sushi on a wooden table",
    "a yellow flower blooming in a garden",
    "a blue bicycle leaning against a wall",
    "a brown horse grazing in a meadow",
    "a purple butterfly on a pink flower",
    "a black cat sleeping on a windowsill",
    "a rainbow appearing after rain",
    "a small boat sailing on the ocean",
    "a cup of hot coffee with steam"
]

# Строим FAISS индекс
text_embeddings = st_model.encode(descriptions, convert_to_numpy=True)  # Эмбеддинги
faiss.normalize_L2(text_embeddings)  # Нормализация
index = faiss.IndexFlatIP(text_embeddings.shape[1])  # Индекс
index.add(text_embeddings)  # Добавляем векторы

# Модели запросов
class SearchRequest(BaseModel):  # Модель поискового запроса
    query: str  # Текст запроса
    top_k: int = 5  # Количество результатов

class SimilarityRequest(BaseModel):  # Модель запроса сходства
    image_base64: str  # Изображение в Base64
    text: str  # Текст для сравнения

class CaptionRequest(BaseModel):  # Модель запроса описания
    image_base64: str  # Изображение в Base64

# Эндпоинт: поиск изображений
@app.post("/search")  # POST /search
def search_images(request: SearchRequest):
    query_emb = st_model.encode([request.query], convert_to_numpy=True)  # Эмбеддинг запроса
    faiss.normalize_L2(query_emb)  # Нормализация
    scores, indices = index.search(query_emb, request.top_k)  # Поиск
    results = []  # Результаты
    for score, idx in zip(scores[0], indices[0]):  # Для каждого результата
        results.append({  # Добавляем
            "id": int(idx),  # ID
            "description": descriptions[idx],  # Описание
            "score": float(score)  # Оценка
        })
    return {"query": request.query, "results": results}  # Ответ

# Эндпоинт: генерация описания
@app.post("/caption")  # POST /caption
def caption_image(request: CaptionRequest):
    try:  # Пробуем обработать
        image_bytes = base64.b64decode(request.image_base64)  # Декодируем Base64
        image = Image.open(io.BytesIO(image_bytes)).convert("RGB")  # Открываем изображение
    except Exception as e:  # При ошибке
        raise HTTPException(status_code=400, detail=f"Invalid image: {str(e)}")  # Ошибка 400
    inputs = blip_processor(image, return_tensors="pt").to(device)  # Токенизация
    with torch.no_grad():  # Без градиентов
        output = blip_model.generate(**inputs, max_new_tokens=50)  # Генерация
    caption = blip_processor.decode(output[0], skip_special_tokens=True)  # Декодируем
    return {"caption": caption}  # Возвращаем описание

# Эндпоинт: оценка сходства
@app.post("/similarity")  # POST /similarity
def compute_similarity(request: SimilarityRequest):
    try:  # Пробуем обработать
        image_bytes = base64.b64decode(request.image_base64)  # Декодируем
        image = Image.open(io.BytesIO(image_bytes)).convert("RGB")  # Открываем
    except Exception as e:  # При ошибке
        raise HTTPException(status_code=400, detail=f"Invalid image: {str(e)}")  # Ошибка
    # Вычисляем эмбеддинг изображения через BLIP (используем визуальный энкодер)
    img_inputs = blip_processor(image, return_tensors="pt").to(device)  # Токенизация
    with torch.no_grad():  # Без градиентов
        # Используем текстовый эмбеддинг описания как приближение
        caption_output = blip_model.generate(**img_inputs, max_new_tokens=30)  # Генерируем описание
        caption = blip_processor.decode(caption_output[0], skip_special_tokens=True)  # Декодируем
    # Вычисляем текст-текст сходство через sentence-transformers
    emb_caption = st_model.encode([caption], convert_to_numpy=True)  # Эмбеддинг описания
    emb_query = st_model.encode([request.text], convert_to_numpy=True)  # Эмбеддинг запроса
    faiss.normalize_L2(emb_caption)  # Нормализация
    faiss.normalize_L2(emb_query)  # Нормализация
    similarity = float(np.dot(emb_caption, emb_query.T)[0, 0])  # Косинусное сходство
    return {  # Возвращаем результат
        "generated_caption": caption,  # Сгенерированное описание
        "query_text": request.text,  # Текст запроса
        "similarity": similarity  # Оценка сходства
    }

# Эндпоинт: проверка здоровья
@app.get("/health")  # GET /health
def health():
    return {"status": "ok", "index_size": index.ntotal}  # Статус
'''

# Записываем сервер в файл
with open("multimodal_server.py", "w") as f:  # Открываем файл
    f.write(server_code)  # Записываем код
print("Сервер сохранён в multimodal_server.py")  # Готово

In [ ]:
# Запускаем сервер с ngrok
import subprocess  # Для запуска процессов
import time  # Для задержек
import threading  # Для потоков
import requests  # Для HTTP запросов

# Функция для запуска uvicorn в фоне
def run_server():
    subprocess.run(["uvicorn", "multimodal_server:app", "--host", "0.0.0.0", "--port", "8000"])  # Запуск

# Запускаем сервер в отдельном потоке
server_thread = threading.Thread(target=run_server, daemon=True)  # Поток-демон
server_thread.start()  # Запускаем
time.sleep(5)  # Ждём старта сервера
print("Сервер запущен на порту 8000")  # Сообщение

# Настройка ngrok для публичного доступа
from pyngrok import ngrok  # Импортируем ngrok
public_url = ngrok.connect(8000)  # Создаём туннель
print(f"Публичный URL: {public_url}")  # Выводим URL

In [ ]:
# Тестируем все эндпоинты сервера

# 1. Проверка здоровья
try:  # Пробуем
    health_resp = requests.get(f"{public_url}/health", timeout=10)  # GET /health
    print(f"Health: {health_resp.json()}")  # Выводим статус
except Exception as e:  # При ошибке
    print(f"Health check failed: {e}")  # Сообщение об ошибке

# 2. Поиск изображений
try:  # Пробуем
    search_resp = requests.post(  # POST /search
        f"{public_url}/search",
        json={"query": "cute animal", "top_k": 3},
        timeout=15
    )
    print(f"\nSearch results:")  # Заголовок
    for r in search_resp.json()["results"]:  # Для каждого результата
        print(f"  [{r['score']:.3f}] {r['description']}")  # Выводим
except Exception as e:  # При ошибке
    print(f"Search failed: {e}")  # Сообщение

# 3. Генерация описания (с тестовым изображением)
try:  # Пробуем
    # Кодируем тестовое изображение в Base64
    test_img = create_test_image("cat")  # Создаём изображение
    buf = io.BytesIO()  # Буфер
    test_img.save(buf, format="JPEG")  # Сохраняем JPEG
    img_b64 = base64.b64encode(buf.getvalue()).decode("utf-8")  # Base64

    caption_resp = requests.post(  # POST /caption
        f"{public_url}/caption",
        json={"image_base64": img_b64},
        timeout=30
    )
    print(f"\nCaption: {caption_resp.json()}")  # Результат
except Exception as e:  # При ошибке
    print(f"Caption failed: {e}")  # Сообщение

# 4. Оценка сходства
try:  # Пробуем
    sim_resp = requests.post(  # POST /similarity
        f"{public_url}/similarity",
        json={"image_base64": img_b64, "text": "a cat sitting on furniture"},
        timeout=30
    )
    print(f"\nSimilarity: {sim_resp.json()}")  # Результат
except Exception as e:  # При ошибке
    print(f"Similarity failed: {e}")  # Сообщение

In [ ]:
# Curl-команды для тестирования (скопируйте и выполните в терминале)
import base64  # Для кодирования
import urllib.parse  # Для URL

# Формируем curl-команды
print("=== Curl-команды для тестирования ===\n")  # Заголовок

# Подготавливаем Base64 изображение для curl
test_img_for_curl = create_test_image("sunset")  # Создаём изображение
buf = io.BytesIO()  # Буфер
test_img_for_curl.save(buf, format="JPEG")  # Сохраняем
img_b64_curl = base64.b64encode(buf.getvalue()).decode("utf-8")  # Base64

# 1. Health check
print("# 1. Проверка здоровья сервера:")  # Комментарий
print(f"curl {public_url}/health\n")  # Команда

# 2. Поиск
print("# 2. Поиск изображений:")  # Комментарий
print(f"curl -X POST {public_url}/search \")  # Команда
print(f'  -H "Content-Type: application/json" \')  # Заголовок
print(f'  -d '{{"query": "beautiful nature", "top_k": 3}}'\n')  # Тело

# 3. Описание изображения
print("# 3. Генерация описания (нужен Base64 изображения):")  # Комментарий
print(f"curl -X POST {public_url}/caption \")  # Команда
print(f'  -H "Content-Type: application/json" \')  # Заголовок
print(f'  -d '{{"image_base64": "<YOUR_BASE64_IMAGE>"}}'\n')  # Тело

# 4. Сходство
print("# 4. Оценка сходства:")  # Комментарий
print(f"curl -X POST {public_url}/similarity \")  # Команда
print(f'  -H "Content-Type: application/json" \')  # Заголовок
print(f'  -d '{{"image_base64": "<YOUR_BASE64_IMAGE>", "text": "a sunset photo"}}'')  # Тело

---
## Шаг 9. Интерактивный мультимодальный эксплорер

Исследуйте мультимодальный поиск с помощью виджетов:
- **Similarity threshold** - минимальный порог сходства
- **Top-k** - количество результатов
- **Temperature** - параметр для генерации описания
- Визуализация результатов в реальном времени


In [ ]:
# Интерактивный мультимодальный эксплорер с ipywidgets
import ipywidgets as widgets  # Виджеты
from IPython.display import display, clear_output  # Отображение

# Создаём виджеты
query_input = widgets.Text(  # Текстовое поле для запроса
    value="cute animal",  # Значение по умолчанию
    description="Запрос:",  # Подпись
    layout=widgets.Layout(width="400px")  # Ширина
)

threshold_slider = widgets.FloatSlider(  # Слайдер порога сходства
    value=0.3,  # Значение по умолчанию
    min=0.0,  # Минимум
    max=1.0,  # Максимум
    step=0.05,  # Шаг
    description="Порог:",  # Подпись
    continuous_update=False  # Обновление при отпускании
)

top_k_slider = widgets.IntSlider(  # Слайдер количества результатов
    value=5,  # Значение по умолчанию
    min=1,  # Минимум
    max=10,  # Максимум
    step=1,  # Шаг
    description="Top-k:",  # Подпись
    continuous_update=False  # Обновление при отпускании
)

temperature_slider = widgets.FloatSlider(  # Слайдер температуры
    value=1.0,  # Значение по умолчанию
    min=0.1,  # Минимум
    max=2.0,  # Максимум
    step=0.1,  # Шаг
    description="Температура:",  # Подпись
    continuous_update=False  # Обновление при отпускании
)

search_button = widgets.Button(  # Кнопка поиска
    description="Искать!",  # Текст кнопки
    button_style="success",  # Зелёная кнопка
    layout=widgets.Layout(width="200px")  # Ширина
)

output_area = widgets.Output()  # Область вывода результатов

# Функция обработки поиска
def on_search_click(b):  # Обработчик нажатия
    with output_area:  # В области вывода
        clear_output(wait=True)  # Очищаем
        query = query_input.value  # Получаем запрос
        threshold = threshold_slider.value  # Получаем порог
        top_k = top_k_slider.value  # Получаем top_k
        temp = temperature_slider.value  # Получаем температуру

        # Ищем изображения
        query_emb = st_model.encode([query], convert_to_numpy=True)  # Эмбеддинг запроса
        faiss.normalize_L2(query_emb)  # Нормализация
        scores, indices = index.search(query_emb, top_k)  # Поиск

        # Фильтруем по порогу
        valid_results = []  # Результаты выше порога
        for score, idx in zip(scores[0], indices[0]):  # Для каждого
            if score >= threshold:  # Если выше порога
                valid_results.append((score, idx))  # Добавляем

        if not valid_results:  # Если нет результатов
            print(f"Нет результатов выше порога {threshold:.2f}")  # Сообщение
            return  # Выход

        # Визуализируем
        n = len(valid_results)  # Количество
        fig, axes = plt.subplots(1, n, figsize=(3*n, 3))  # Графики
        if n == 1:  # Если один
            axes = [axes]  # Оборачиваем
        for i, (score, idx) in enumerate(valid_results):  # Для каждого
            axes[i].imshow(images_dataset[int(idx)])  # Показываем
            axes[i].set_title(f"Score: {score:.3f}\nT: {temp}", fontsize=9)  # Заголовок
            axes[i].axis("off")  # Убираем оси
        plt.suptitle(f"Запрос: '{query}' (порог={threshold:.2f}, top_k={top_k})",
                     fontsize=12, fontweight="bold")  # Общий заголовок
        plt.tight_layout()  # Подгонка
        plt.show()  # Показываем

        # Генерируем описание для лучшего результата
        best_idx = int(valid_results[0][1])  # Индекс лучшего
        print(f"\nЛучшее совпадение: {descriptions[best_idx]}")  # Описание

# Привязываем обработчик
search_button.on_click(on_search_click)  # Привязка

# Отображаем виджеты
display(widgets.VBox([  # Вертикальная компоновка
    widgets.HBox([query_input, search_button]),  # Запрос + кнопка
    widgets.HBox([threshold_slider, top_k_slider]),  # Слайдеры порога и top_k
    temperature_slider,  # Слайдер температуры
    output_area  # Область вывода
]))

In [ ]:
# Интерактивная визуализация сходства изображений и текстов
sim_query_input = widgets.Text(  # Поле для запроса сходства
    value="a cat",  # По умолчанию
    description="Текст:",  # Подпись
    layout=widgets.Layout(width="400px")  # Ширина
)

sim_button = widgets.Button(  # Кнопка
    description="Показать сходство",  # Текст
    button_style="info"  # Стиль
)

sim_output = widgets.Output()  # Область вывода

def on_sim_click(b):  # Обработчик
    with sim_output:  # В области вывода
        clear_output(wait=True)  # Очищаем
        query = sim_query_input.value  # Получаем текст
        query_emb = st_model.encode([query], convert_to_numpy=True)  # Эмбеддинг
        faiss.normalize_L2(query_emb)  # Нормализация

        # Вычисляем сходства со всеми описаниями
        sims = []  # Список сходств
        for desc in descriptions:  # Для каждого описания
            desc_emb = st_model.encode([desc], convert_to_numpy=True)  # Эмбеддинг описания
            faiss.normalize_L2(desc_emb)  # Нормализация
            sim = float(np.dot(query_emb, desc_emb.T)[0, 0])  # Косинусное сходство
            sims.append(sim)  # Добавляем

        # Визуализируем как бар-чарт
        fig, ax = plt.subplots(figsize=(12, 5))  # График
        short_descs = [d[:30]+"..." if len(d)>30 else d for d in descriptions]  # Короткие описания
        colors = ["green" if s > 0.5 else "orange" if s > 0.3 else "red" for s in sims]  # Цвета
        bars = ax.barh(range(len(sims)), sims, color=colors)  # Горизонтальные бары
        ax.set_yticks(range(len(sims)))  # Деления Y
        ax.set_yticklabels(short_descs, fontsize=8)  # Подписи Y
        ax.set_xlabel("Косинусное сходство", fontsize=11)  # Подпись X
        ax.set_title(f"Сходство с запросом: '{query}'", fontsize=13, fontweight="bold")  # Заголовок
        ax.axvline(x=0.5, color="green", linestyle="--", alpha=0.5, label="Порог 0.5")  # Порог
        ax.legend()  # Легенда
        plt.tight_layout()  # Подгонка
        plt.show()  # Показываем

sim_button.on_click(on_sim_click)  # Привязка

display(widgets.VBox([  # Отображаем
    widgets.HBox([sim_query_input, sim_button]),  # Запрос + кнопка
    sim_output  # Вывод
]))

---
## Шаг 10. Cross-attention: как зрение встречается с языком

**Cross-attention** - механизм, позволяющий текстовому декодеру
"смотреть" на различные части изображения при генерации каждого слова.

Когда модель пишет "кот", она обращает внимание на область с котом.
Когда пишет "диван" - на область с диваном.

Мы реализуем упрощённый cross-attention и визуализируем карты внимания.


In [ ]:
# Реализация cross-attention с нуля
class CrossAttention(nn.Module):
    # Cross-attention: Query из одного модальности, Key/Value из другой
    def __init__(self, dim_q: int, dim_kv: int, n_heads: int = 4):
        super().__init__()  # Инициализация
        self.n_heads = n_heads  # Количество голов внимания
        self.head_dim = dim_q // n_heads  # Размерность каждой головы

        # Линейные проекции для Q, K, V
        self.W_q = nn.Linear(dim_q, dim_q)  # Query проекция
        self.W_k = nn.Linear(dim_kv, dim_q)  # Key проекция
        self.W_v = nn.Linear(dim_kv, dim_q)  # Value проекция
        self.W_out = nn.Linear(dim_q, dim_q)  # Выходная проекция
        self.scale = self.head_dim ** -0.5  # Масштабирование

    def forward(self, query, key_value, return_attention=False):
        # query: [batch, seq_q, dim_q] - из текстового декодера
        # key_value: [batch, seq_kv, dim_kv] - из визуального энкодера
        batch_size = query.shape[0]  # Размер батча

        # Проецируем Q, K, V
        Q = self.W_q(query)  # [batch, seq_q, dim_q]
        K = self.W_k(key_value)  # [batch, seq_kv, dim_q]
        V = self.W_v(key_value)  # [batch, seq_kv, dim_q]

        # Разбиваем на головы: [batch, n_heads, seq, head_dim]
        Q = Q.view(batch_size, -1, self.n_heads, self.head_dim).transpose(1, 2)  # Разбиваем Q
        K = K.view(batch_size, -1, self.n_heads, self.head_dim).transpose(1, 2)  # Разбиваем K
        V = V.view(batch_size, -1, self.n_heads, self.head_dim).transpose(1, 2)  # Разбиваем V

        # Вычисляем attention scores
        scores = torch.matmul(Q, K.transpose(-2, -1)) * self.scale  # [batch, n_heads, seq_q, seq_kv]
        attn_weights = F.softmax(scores, dim=-1)  # Нормализуем по последней оси

        # Применяем внимание к Values
        context = torch.matmul(attn_weights, V)  # [batch, n_heads, seq_q, head_dim]

        # Объединяем головы
        context = context.transpose(1, 2).contiguous()  # [batch, seq_q, n_heads, head_dim]
        context = context.view(batch_size, -1, self.n_heads * self.head_dim)  # [batch, seq_q, dim_q]

        # Выходная проекция
        output = self.W_out(context)  # [batch, seq_q, dim_q]

        if return_attention:  # Если нужны веса внимания
            return output, attn_weights  # Возвращаем и output, и attention
        return output  # Иначе только output

# Тестируем cross-attention
cross_attn = CrossAttention(dim_q=64, dim_kv=128, n_heads=4)  # Создаём слой
test_query = torch.randn(1, 10, 64)  # Текстовые фичи: 10 токенов
test_kv = torch.randn(1, 49, 128)  # Визуальные фичи: 7x7=49 патчей
output, attn = cross_attn(test_query, test_kv, return_attention=True)  # Прямой проход
print(f"Query: {test_query.shape}")  # Размерность запроса
print(f"Key/Value: {test_kv.shape}")  # Размерность KV
print(f"Output: {output.shape}")  # Размерность выхода
print(f"Attention: {attn.shape}")  # Размерность attention

In [ ]:
# Визуализация карт cross-attention
# Симулируем внимание текстовых токенов к пространственным областям изображения

fig, axes = plt.subplots(2, 5, figsize=(18, 7))  # 2 строки x 5 столбцов

# Слова генерируемого описания
words = ["a", "fluffy", "orange", "cat", "sitting", "on", "a", "brown", "sofa", "at"]  # 10 слов

# Создаём симулированные карты внимания для каждого слова
np.random.seed(42)  # Фиксируем случайность
for i, word in enumerate(words):  # Для каждого слова
    row = i // 5  # Строка
    col = i % 5  # Столбец
    ax = axes[row, col]  # Текущий подграфик

    # Создаём карту внимания 7x7 (как 7x7 патчей изображения)
    if word in ["cat", "fluffy", "orange"]:  # Слова про кота
        # Внимание на центр-верх (где кот)
        attn_map = np.random.rand(7, 7) * 0.3  # Фон
        attn_map[1:4, 2:5] = np.random.rand(3, 3) * 0.3 + 0.7  # Кот (центр)
    elif word in ["sofa", "brown"]:  # Слова про диван
        # Внимание на нижнюю часть (где диван)
        attn_map = np.random.rand(7, 7) * 0.3  # Фон
        attn_map[4:7, 1:6] = np.random.rand(3, 5) * 0.3 + 0.7  # Диван (низ)
    elif word == "sitting":  # Слово про действие
        # Внимание на переходную зону
        attn_map = np.random.rand(7, 7) * 0.3  # Фон
        attn_map[3:5, 2:5] = np.random.rand(2, 3) * 0.3 + 0.6  # Переход
    else:  # Остальные слова
        attn_map = np.random.rand(7, 7) * 0.5  # Размытое внимание

    # Нормализуем карту внимания
    attn_map = attn_map / attn_map.sum()  # Сумма = 1

    # Отрисовываем тепловую карту
    im = ax.imshow(attn_map, cmap="hot", interpolation="bilinear")  # Тепловая карта
    ax.set_title(f'"{word}"', fontsize=12, fontweight="bold")  # Слово как заголовок
    ax.axis("off")  # Убираем оси

plt.suptitle("Cross-Attention: на какую часть изображения смотрит модель\nпри генерации каждого слова",
             fontsize=14, fontweight="bold")  # Общий заголовок
plt.tight_layout()  # Подгонка
plt.colorbar(im, ax=axes, shrink=0.6, label="Вес внимания")  # Общая цветовая шкала
plt.show()  # Показываем

In [ ]:
# Более детальная визуализация: attention на реальном изображении
# Показываем, как разные слова "подсвечивают" разные области

fig, axes = plt.subplots(2, 4, figsize=(16, 8))  # 2x4 подграфиков

# Создаём тестовое изображение "кот на диване"
base_img = create_test_image("cat")  # Изображение кота
base_array = np.array(base_img)  # Как массив NumPy

# Слова для визуализации
focus_words = ["cat", "orange", "fluffy", "sitting", "sofa", "brown", "indoor", "cozy"]  # 8 слов

for i, word in enumerate(focus_words):  # Для каждого слова
    row = i // 4  # Строка
    col = i % 4  # Столбец
    ax = axes[row, col]  # Подграфик

    # Показываем исходное изображение
    ax.imshow(base_array)  # Изображение

    # Создаём полупрозрачную маску внимания
    h, w = base_array.shape[:2]  # Размеры
    mask = np.zeros((h, w))  # Нулевая маска

    if word in ["cat", "orange", "fluffy"]:  # Фокус на коте
        # Центральная верхняя область
        mask[h//4:h//2, w//4:3*w//4] = 0.7  # Кот
        mask = mask + np.random.rand(h, w) * 0.1  # Шум
    elif word in ["sofa", "brown"]:  # Фокус на диване
        mask[h//2:3*h//4, w//8:7*w//8] = 0.7  # Диван
        mask = mask + np.random.rand(h, w) * 0.1  # Шум
    elif word == "sitting":  # Переходная зона
        mask[h//3:2*h//3, w//4:3*w//4] = 0.5  # Переход
        mask = mask + np.random.rand(h, w) * 0.15  # Шум
    elif word == "indoor":  # Всё помещение
        mask[:, :] = 0.3  # Размыто
        mask[h//4:3*h//4, w//8:7*w//8] = 0.5  # Внутри
    else:  # cozy - тёплое чувство
        mask[:, :] = 0.2  # Очень размыто
        mask[h//4:3*h//4, w//4:3*w//4] = 0.4  # Центр

    # Накладываем маску
    ax.imshow(mask, cmap="Reds", alpha=0.5, interpolation="bilinear")  # Полупрозрачная маска
    ax.set_title(f'"{word}"', fontsize=13, fontweight="bold", color="darkred")  # Заголовок
    ax.axis("off")  # Убираем оси

plt.suptitle("Cross-Attention визуализация: красным подсвечены области внимания",
             fontsize=14, fontweight="bold")  # Общий заголовок
plt.tight_layout()  # Подгонка
plt.show()  # Показываем

---
## Шаг 11. Аудио + текст: основы распознавания речи

**Speech-to-Text** (ASR) - ещё один вид мультимодальности: аудио -> текст.

Мы создадим простую симуляцию ASR пайплайна и визуализируем
как звуковые волны превращаются в текст.


In [ ]:
# Симуляция Audio-to-Text пайплайна
# В реальном Colab можно использовать whisper или wav2vec2

# Генерируем синтетический аудио-сигнал (имитация речи)
sample_rate = 16000  # Частота дискретизации (16 кГц)
duration = 2.0  # Длительность в секундах
t = np.linspace(0, duration, int(sample_rate * duration))  # Временная ось

# Создаём сигнал, имитирующий речь с разными частотами для разных "слов"
# "Слово 1": низкая частота (гласные)
word1 = 0.5 * np.sin(2 * np.pi * 200 * t[:8000])  # Низкочастотный сигнал
# Пауза
pause1 = np.zeros(2000)  # Тишина между словами
# "Слово 2": средняя частота (согласные)
word2 = 0.3 * np.sin(2 * np.pi * 800 * t[8000:16000])  # Среднечастотный сигнал
# Пауза
pause2 = np.zeros(2000)  # Тишина
# "Слово 3": высокая частота (шипящие)
word3 = 0.2 * np.random.randn(10000)  # Шумовой сигнал (шипящие)
# Склеиваем
audio_signal = np.concatenate([word1, pause1, word2, pause2, word3])  # Полный сигнал
audio_signal = audio_signal / np.max(np.abs(audio_signal))  # Нормализация

# Визуализируем звуковую волну
fig, axes = plt.subplots(3, 1, figsize=(14, 8))  # 3 графика

# 1. Временная область (waveform)
axes[0].plot(t[:len(audio_signal)], audio_signal, linewidth=0.5, color="steelblue")  # Волна
axes[0].set_title("Аудиосигнал (временная область)", fontsize=12, fontweight="bold")  # Заголовок
axes[0].set_xlabel("Время (с)")  # Подпись X
axes[0].set_ylabel("Амплитуда")  # Подпись Y
axes[0].axvspan(0, 0.5, alpha=0.2, color="green", label="Слово 1")  # Выделение слова 1
axes[0].axvspan(0.625, 1.125, alpha=0.2, color="blue", label="Слово 2")  # Выделение слова 2
axes[0].axvspan(1.375, 2.0, alpha=0.2, color="red", label="Слово 3")  # Выделение слова 3
axes[0].legend(fontsize=9)  # Легенда

# 2. Спектрограмма (частотно-временное представление)
from scipy.signal import spectrogram as scipy_spectrogram  # Спектрограмма
try:  # Пробуем scipy
    f, t_spec, Sxx = scipy_spectrogram(audio_signal, fs=sample_rate, nperseg=256)  # Спектрограмма
    axes[1].pcolormesh(t_spec, f[:50], 10*np.log10(Sxx[:50]+1e-10),
                       shading="gouraud", cmap="viridis")  # Тепловая карта
except ImportError:  # Если нет scipy
    # Простая спектрограмма через STFT
    from numpy.fft import fft  # FFT
    window_size = 256  # Размер окна
    hop_size = 128  # Шаг окна
    n_windows = (len(audio_signal) - window_size) // hop_size  # Количество окон
    spec = []  # Спектрограмма
    for w in range(n_windows):  # Для каждого окна
        start = w * hop_size  # Начало
        segment = audio_signal[start:start+window_size]  # Сегмент
        spectrum = np.abs(fft(segment))[:window_size//2]  # FFT (одна сторона)
        spec.append(spectrum)  # Добавляем
    spec = np.array(spec).T  # Транспонируем
    axes[1].imshow(10*np.log10(spec+1e-10), aspect="auto", origin="lower",
                   cmap="viridis")  # Тепловая карта

axes[1].set_title("Спектрограмма (частотно-временное представление)", fontsize=12, fontweight="bold")  # Заголовок
axes[1].set_ylabel("Частота (Гц)")  # Подпись Y
axes[1].set_xlabel("Время (с)")  # Подпись X

# 3. Результат "распознавания" - какие слова найдены
simulated_words = [  # Симулированный результат ASR
    {"word": "Привет", "start": 0.0, "end": 0.5, "confidence": 0.95},
    {"word": "мир", "start": 0.625, "end": 1.125, "confidence": 0.88},
    {"word": "ИИ", "start": 1.375, "end": 2.0, "confidence": 0.82}
]

axes[2].set_xlim(0, 2.0)  # Пределы X
axes[2].set_ylim(0, 1)  # Пределы Y
axes[2].set_title("Результат распознавания речи (ASR)", fontsize=12, fontweight="bold")  # Заголовок
colors_asr = ["green", "blue", "red"]  # Цвета для слов
for i, w in enumerate(simulated_words):  # Для каждого слова
    axes[2].barh(0.5, w["end"]-w["start"], left=w["start"], height=0.4,
                 color=colors_asr[i], alpha=0.6,
                 label=f'{w["word"]} ({w["confidence"]:.0%})')  # Блок слова
    axes[2].text((w["start"]+w["end"])/2, 0.5, w["word"],
                 ha="center", va="center", fontsize=12, fontweight="bold")  # Текст слова
axes[2].set_xlabel("Время (с)")  # Подпись X
axes[2].legend(fontsize=9, loc="upper right")  # Легенда
axes[2].set_yticks([])  # Убираем ось Y

plt.tight_layout()  # Подгонка
plt.show()  # Показываем

print("\nСимулированный результат ASR:")  # Заголовок
for w in simulated_words:  # Для каждого слова
    print(f"  [{w['start']:.2f}-{w['end']:.2f}s] {w['word']} (уверенность: {w['confidence']:.0%})")  # Вывод

In [ ]:
# Попробуем реальный ASR если доступен микрофон или аудиофайл
# Для Colab можно загрузить аудиофайл или использовать whisper

try:  # Пробуем импортировать whisper
    import whisper  # OpenAI Whisper для ASR
    whisper_available = True  # Флаг доступности
    print("Whisper доступен!")  # Сообщение
except ImportError:  # Если не установлен
    whisper_available = False  # Флаг
    print("Whisper не установлен. Установим его...")  # Сообщение
    try:  # Пробуем установить
        import subprocess  # Для pip
        subprocess.run(["pip", "install", "-q", "openai-whisper"], check=True)  # Установка
        import whisper  # Импортируем после установки
        whisper_available = True  # Флаг
        print("Whisper установлен!")  # Сообщение
    except Exception as e:  # При ошибке
        print(f"Не удалось установить Whisper: {e}")  # Сообщение
        print("Продолжаем с симуляцией ASR.")  # Сообщение

if whisper_available:  # Если whisper доступен
    # Загружаем маленькую модель для быстрой работы
    asr_model = whisper.load_model("tiny")  # Маленькая модель
    print(f"Модель Whisper 'tiny' загружена")  # Сообщение
    print("Вы можете загрузить аудиофайл и распознать его:")  # Подсказка
    print("  result = asr_model.transcribe('audio.mp3', language='ru')")  # Пример
    print("  print(result['text'])")  # Вывод результата
else:  # Если whisper не доступен
    # Демонстрируем пайплайн ASR визуально
    print("\n=== Симуляция ASR пайплайна ===")  # Заголовок
    print("1. Аудиосигнал -> Спектрограмма (FFT)")  # Шаг 1
    print("2. Спектрограмма -> Энкодер (CNN + Transformer)")  # Шаг 2
    print("3. Энкодер -> Декодер (Transformer) -> Текст")  # Шаг 3
    print("\nМодели ASR: Whisper, Wav2Vec2, DeepSpeech")  # Список моделей

---
## Шаг 12. Ландшафт мультимодальных моделей

Сравним ключевые мультимодальные модели:

| Модель | Год | Модальности | Ключевая идея | Размер |
|--------|-----|-------------|---------------|--------|
| **CLIP** | 2021 | Image + Text | Контрастивное обучение | 400M |
| **BLIP** | 2022 | Image + Text | Bootstrapping + Captioning | 385M |
| **BLIP-2** | 2023 | Image + Text | Q-Former мост | 3.4B |
| **LLaVA** | 2023 | Image + Text | LLM + Visual Encoder | 7-13B |
| **GPT-4V** | 2023 | Image + Text | Масштабный мультимодальный LLM | ~1T* |
| **Gemini** | 2023 | Image+Text+Audio+Video | Нативная мультимодальность | ~600B* |
| **Whisper** | 2022 | Audio -> Text | ASR на 99 языках | 1.5B |


In [ ]:
# Визуализация: сравнение мультимодальных моделей
fig, axes = plt.subplots(1, 2, figsize=(16, 7))  # 2 графика

# --- Левый график: поддерживаемые модальности ---
ax1 = axes[0]  # Первый подграфик
models = ["CLIP", "BLIP", "BLIP-2", "LLaVA", "GPT-4V", "Gemini", "Whisper"]  # Модели
modalities = {  # Поддерживаемые модальности
    "Image": [1, 1, 1, 1, 1, 1, 0],  # Работа с изображениями
    "Text": [1, 1, 1, 1, 1, 1, 0],  # Работа с текстом
    "Img->Text": [0, 1, 1, 1, 1, 1, 0],  # Генерация текста по картинке
    "Text->Img": [0, 0, 0, 0, 0, 0, 0],  # Генерация изображений
    "Audio": [0, 0, 0, 0, 0, 1, 1],  # Работа с аудио
    "Video": [0, 0, 0, 0, 0, 1, 0]  # Работа с видео
}

modality_names = list(modalities.keys())  # Названия модальностей
x = np.arange(len(models))  # Позиции по X
width = 0.12  # Ширина столбца

colors_mod = ["#e74c3c", "#3498db", "#2ecc71", "#9b59b6", "#f39c12", "#1abc9c"]  # Цвета

for i, mod_name in enumerate(modality_names):  # Для каждой модальности
    values = modalities[mod_name]  # Значения
    offset = (i - len(modality_names)/2) * width  # Смещение
    ax1.bar(x + offset, values, width, label=mod_name, color=colors_mod[i], alpha=0.8)  # Столбцы

ax1.set_xlabel("Модель", fontsize=11)  # Подпись X
ax1.set_ylabel("Поддержка", fontsize=11)  # Подпись Y
ax1.set_title("Поддерживаемые модальности", fontsize=13, fontweight="bold")  # Заголовок
ax1.set_xticks(x)  # Деления
ax1.set_xticklabels(models, fontsize=9, rotation=30)  # Подписи
ax1.legend(fontsize=8, loc="upper left")  # Легенда
ax1.set_ylim(0, 1.5)  # Пределы Y
ax1.set_yticks([0, 1])  # Только 0 и 1
ax1.set_yticklabels(["Нет", "Да"])  # Подписи

# --- Правый график: размер и возможности ---
ax2 = axes[1]  # Второй подграфик
model_params = [0.4, 0.385, 3.4, 10, 1000, 600, 1.5]  # Параметры в миллиардах (log scale)
# Возможности: рейтинг от 1 до 5
capabilities = [4, 4, 5, 5, 5, 5, 3]  # Общая оценка возможностей

scatter_colors = ["#e74c3c", "#3498db", "#2ecc71", "#9b59b6", "#f1c40f", "#1abc9c", "#e67e22"]  # Цвета

for i, model_name in enumerate(models):  # Для каждой модели
    ax2.scatter(np.log10(model_params[i]), capabilities[i],
                s=200, c=scatter_colors[i], zorder=5, edgecolors="black", linewidth=1.5)  # Точка
    ax2.annotate(model_name, (np.log10(model_params[i]), capabilities[i]),
                 fontsize=9, ha="center", va="bottom", fontweight="bold")  # Подпись

ax2.set_xlabel("log10(Параметры, млрд)", fontsize=11)  # Подпись X
ax2.set_ylabel("Возможности (1-5)", fontsize=11)  # Подпись Y
ax2.set_title("Размер модели vs Возможности", fontsize=13, fontweight="bold")  # Заголовок
ax2.grid(True, alpha=0.3)  # Сетка

plt.tight_layout()  # Подгонка
plt.show()  # Показываем

In [ ]:
# Визуализация: timeline развития мультимодальных моделей
fig, ax = plt.subplots(figsize=(14, 6))  # Большой график

# Данные: модель, год, ключевой вклад
timeline = [
    (2012, "AlexNet", "Революция в зрении", "blue"),
    (2017, "Transformer", "Attention is all you need", "green"),
    (2018, "BERT", "Предобучение языка", "orange"),
    (2020, "ViT", "Transformer для изображений", "purple"),
    (2021, "CLIP", "Image-Text контраст", "red"),
    (2021, "DALL-E", "Text->Image генерация", "pink"),
    (2022, "BLIP", "Загрузка + генерация", "cyan"),
    (2022, "Whisper", "ASR на 99 языках", "brown"),
    (2022, "Stable Diffusion", "Открытая генерация изображений", "magenta"),
    (2023, "BLIP-2", "Q-Former мост", "teal"),
    (2023, "LLaVA", "LLM + Visual Encoder", "navy"),
    (2023, "GPT-4V", "Масштабный VLM", "gold"),
    (2023, "Gemini", "Нативная мультимодальность", "darkgreen"),
    (2024, "GPT-4o", "Омнимодальность", "crimson"),
]

# Рисуем timeline
for i, (year, name, contribution, color) in enumerate(timeline):  # Для каждого события
    y_pos = 0.5 + (i % 2) * 0.3  # Чередуем позиции Y
    ax.scatter(year, y_pos, s=200, c=color, zorder=5, edgecolors="black")  # Точка
    ax.annotate(f"{name}\n({contribution})",
                (year, y_pos), fontsize=8, ha="center",
                va="bottom" if i % 2 == 0 else "top",
                fontweight="bold")  # Подпись

# Линия времени
ax.axhline(y=0.5, color="gray", linestyle="-", linewidth=2, alpha=0.3)  # Линия

# Разделяем на эпохи
ax.axvspan(2012, 2019, alpha=0.05, color="blue", label="Эпоха CNN")  # Эпоха CNN
ax.axvspan(2019, 2022, alpha=0.05, color="green", label="Эпоха Transformers")  # Эпоха трансформеров
ax.axvspan(2022, 2025, alpha=0.05, color="red", label="Эпоха мультимодальности")  # Мультимодальность

ax.set_xlim(2011, 2025)  # Пределы X
ax.set_ylim(0, 1.2)  # Пределы Y
ax.set_xlabel("Год", fontsize=12)  # Подпись X
ax.set_title("Timeline развития мультимодального ИИ", fontsize=14, fontweight="bold")  # Заголовок
ax.legend(fontsize=9, loc="upper left")  # Легенда
ax.set_yticks([])  # Убираем ось Y
plt.tight_layout()  # Подгонка
plt.show()  # Показываем

---
## Шаг 13. Практические задания

Теперь ваша очередь! Вот 5 заданий для закрепления материала.

### Задание 1: Улучшите SimpleCLIP

Модифицируйте `SimpleImageEncoder`:
- Добавьте BatchNorm после каждого Conv слоя
- Увеличьте количество каналов (16 -> 32 -> 64 -> 128)
- Добавьте dropout с p=0.3
- Сравните качество (loss после 50 эпох) с базовой версией

### Задание 2: Мультимодальный классификатор

Используя эмбеддинги CLIP (через sentence-transformers):
- Создайте классификатор изображений по текстовым описаниям
- Реализуйте zero-shot классификацию: "это фото кота или собаки?"
- Сравните с обычным классификатором на пикселях

### Задание 3: Расширенный поиск

Улучшите поисковую систему:
- Добавьте фильтрацию по порогу сходства на сервере
- Реализуйте поиск "похожих изображений" (image-to-image)
- Добавьте сортировку по дате или рейтингу

### Задание 4: Кастомный cross-attention

Реализуйте двунаправленный cross-attention:
- Text -> Image attention (текст смотрит на изображение)
- Image -> Text attention (изображение смотрит на текст)
- Объедините оба направления через gating

### Задание 5: Мультимодальный чат-бот

Создайте простой чат-бот, который:
- Принимает изображение и вопрос пользователя
- Генерирует ответ с помощью BLIP-VQA
- Поддерживает историю диалога (контекст)
- Оформите как FastAPI сервер с эндпоинтом `/chat`

---

**Критерии оценки:**
- Код работает без ошибок - 30%
- Код прокомментирован - 20%
- Оригинальность решения - 20%
- Качество визуализации - 15%
- Объяснение результатов - 15%

---

---
## Итог

В этом ноутбуке вы:

1. Поймали суть **мультимодальности** - объединение разных типов данных
2. Разобрали **CLIP** и контрастивное обучение
3. Реализовали **SimpleCLIP с нуля** - энкодеры + contrastive loss
4. Освоили **Image Captioning** с предобученным BLIP
5. Попробовали **Visual Question Answering**
6. Построили **поисковик изображений** с FAISS
7. Запустили **FastAPI сервер** мультимодального поиска
8. Исследовали параметры через **ipywidgets**
9. Визуализировали **cross-attention** карты
10. Познакомились с **ASR** и аудио-модальностью
11. Сравнили **мультимодальные модели** от CLIP до Gemini

### Ключевые выводы

- Мультимодальность = общее пространство эмбеддингов для разных модальностей
- Контрастивное обучение сближает правильные пары и раздвигает неправильные
- Cross-attention позволяет декодеру "смотреть" на нужные части изображения
- FAISS делает поиск по векторам быстрым даже для миллионов элементов
- Предобученные модели (BLIP, CLIP) дают мощный baseline без тренировки

### Что дальше?

- Изучите **LLaVA** и **GPT-4V** для сложных визуальных задач
- Попробуйте **видео-модальность** с моделями вроде Gemini
- Исследуйте **multimodal RLHF** для выравнивания мультимодальных моделей
- Создайте собственный **мультимодальный датасет** и дообучите модель

---